In [8]:
# mask-analog imports
import run_experiments as rexp
# create an Experiments object which allows you to access previous experiments and define new ones (and train a mask)
exp_object = rexp.Experiments('experiments.json')
import plot_analogs as pa
import best_analogs as ba
import numpy as np
# hide some warnings that are not helpful
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
# make plots show up in the notebook
%matplotlib inline

# workflow for a new experiment

#### 1. preprocess data (saves data that is temporally and spatially smoothed)

#### 2. create the settings for a new experiment

#### 3. train (and plot) a mask based on the settings

#### 4. generate (or select a previously generated) analog library

#### 5. find the best analogs for a given state of interest (model, member)

#### 6. plot the results

### 1. preprocess data (saves data that is temporally and spatially smoothed)

In [2]:
# var_list and model_list = -1 means use all available
# scratch means load the models and resolutions by reading the files directly (otherwise, use the version I hand put in there)

# uncomment and run this if you need something that is not already preprocessed -- will take a while (several hours)
# import data_processing as dp
# dp.preprocess_data(resolution, var_list=-1, model_list=-1, overwrite_data=False, scratch=False)

In [10]:
# all_models = [
#     ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
#     ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
#     ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
#     ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
#     ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
#     ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
#     ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'MRI-ESM2-0'],
#     ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'MRI-ESM2-0'],
#     ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'MRI-ESM2-0'],
#     ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'MRI-ESM2-0']
#     ]
# all_models = ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2']
# all_models = ['MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'MRI-ESM2-0', 'GISS-E2-2-G', 'AWI-ESM-1-1']
all_models = ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0']
obs = ['ERA5', 'BEST']

tether = 2
soi_model = 'BEST'
soi_member = 0
regions = ["amazon", "northern europe",  "southern india", "african great lakes"]
region_names = ["amzn", "neur", "sind", "glaf"]
leadtime = 5
for ii, region_name in enumerate(region_names):
    experiment_name = f"{region_name}_lt{leadtime}_best_global1956on"
    new_exp = exp_object.make_exp_dictionary(experiment_name, variable_list=['tas'], analog_models=all_models, soi_models=all_models, leadtime=leadtime, split=[5, 3, 2], region=regions[ii], val_models=all_models, target_var=['tas'], window=False, trajectory=1, resolutions=1, time_range=False, norm_data=True, baseline=[1961, 1990], remove_trend=False, norm_weights=True, transfer_exp=None, mask=False, prediction_nodes=[], prediction_activation='elu', learning_rate=0.001, loss='mse', patience=50, min_delta=0.005, batch_size=64, val_batch_size=2500, max_epochs=5000, mask_activation='relu', mask_init="ones", mask_l1=0.0, mask_l2=0.0, seed=20)
    exp_object.add_experiment(new_exp)
    exp_object.make_mask(experiment_name, overwrite_model=True, overwrite_mask=True)
    library = ba.BestAnalogs(experiment_name, library_name=None, model_list=all_models+['BEST'], n_members=-1)
    library.get_all_soi_analogs(soi_model, soi_member, tether=tether, savename=None, quant=None, mask_type='weights', leadtime=None)

experiment_settings are:
{'amzn_lt5_best_global1956on': {'experiment_name': 'amzn_lt5_best_global1956on', 'variable_list': ['tas'], 'analog_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'soi_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'leadtime': 5, 'split': {'analog': 5, 'training': 3, 'validation': 2}, 'region': 'amazon', 'val_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'target_var': ['tas'], 'window': False, 'trajectory': 1, 'resolutions': [2.7672718509152645, 2.8125], 'time_range': False, 'norm_data': True, 'baseline': [1961, 1990], 'remove_trend': False, 'norm_weights': True, 'transfer_exp': None, 'mask': False, 'prediction_nodes': [], 'prediction_activation': 'elu', 'learning_rate': 0.001, 'loss': 'mse', 'patience': 

100%|██████████| 173/173 [01:30<00:00,  1.90it/s]


saving to: /Users/mafern/Desktop/mask-analog-projections/saved_analogs/amzn_lt5_best_global1956on/BEST_mem0_teth2.pickle
experiment_settings are:
{'neur_lt5_best_global1956on': {'experiment_name': 'neur_lt5_best_global1956on', 'variable_list': ['tas'], 'analog_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'soi_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'leadtime': 5, 'split': {'analog': 5, 'training': 3, 'validation': 2}, 'region': 'northern europe', 'val_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'target_var': ['tas'], 'window': False, 'trajectory': 1, 'resolutions': [2.7672718509152645, 2.8125], 'time_range': False, 'norm_data': True, 'baseline': [1961, 1990], 'remove_trend': False, 'norm_weights': True, 'transfer_exp'

100%|██████████| 173/173 [01:27<00:00,  1.97it/s]


saving to: /Users/mafern/Desktop/mask-analog-projections/saved_analogs/neur_lt5_best_global1956on/BEST_mem0_teth2.pickle
experiment_settings are:
{'sind_lt5_best_global1956on': {'experiment_name': 'sind_lt5_best_global1956on', 'variable_list': ['tas'], 'analog_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'soi_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'leadtime': 5, 'split': {'analog': 5, 'training': 3, 'validation': 2}, 'region': 'southern india', 'val_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'target_var': ['tas'], 'window': False, 'trajectory': 1, 'resolutions': [2.7672718509152645, 2.8125], 'time_range': False, 'norm_data': True, 'baseline': [1961, 1990], 'remove_trend': False, 'norm_weights': True, 'transfer_exp':

100%|██████████| 173/173 [01:29<00:00,  1.94it/s]


saving to: /Users/mafern/Desktop/mask-analog-projections/saved_analogs/sind_lt5_best_global1956on/BEST_mem0_teth2.pickle
experiment_settings are:
{'glaf_lt5_best_global1956on': {'experiment_name': 'glaf_lt5_best_global1956on', 'variable_list': ['tas'], 'analog_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'soi_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'leadtime': 5, 'split': {'analog': 5, 'training': 3, 'validation': 2}, 'region': 'african great lakes', 'val_models': ['MIROC-ES2H', 'MIROC-ES2L', 'TaiESM', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'ACCESS-CM2', 'GISS-E2-2-G', 'MRI-ESM2-0'], 'target_var': ['tas'], 'window': False, 'trajectory': 1, 'resolutions': [2.7672718509152645, 2.8125], 'time_range': False, 'norm_data': True, 'baseline': [1961, 1990], 'remove_trend': False, 'norm_weights': True, 'transfer_

100%|██████████| 173/173 [01:27<00:00,  1.99it/s]


saving to: /Users/mafern/Desktop/mask-analog-projections/saved_analogs/glaf_lt5_best_global1956on/BEST_mem0_teth2.pickle


In [ ]:
# transfer learn the above
all_models = [
    ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
    ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
    ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
    ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
    ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
    ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'],
    ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'MRI-ESM2-0'],
    ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'MRI-ESM2-0'],
    ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'MRI-ESM2-0'],
    ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'MRI-ESM2-0']
    ]
obs = ['ERA5', 'BEST']

tether = 2
soi_model = 'BEST'
soi_member = 0
for ii, leadtime in enumerate([1,2,3,4,5,6,7,8,9,10]):
    experiment_name = f"wus_lt{leadtime}_best_global"
    new_exp = exp_object.make_exp_dictionary(experiment_name+'_tl', variable_list=['tas'], analog_models=['BEST2008'], soi_models=all_models[ii], leadtime=leadtime, split=[1, 3, 2], region='western us', val_models=all_models[ii], target_var=['tas'], window=False, trajectory=1, resolutions=1, time_range=False, norm_data=True, baseline=[1961, 1990], remove_trend=False, norm_weights=True, transfer_exp=experiment_name, mask=False, prediction_nodes=[], prediction_activation='elu', learning_rate=0.0001, loss='mse', patience=500, min_delta=0.005, batch_size=64, val_batch_size=2500, max_epochs=150, mask_activation='relu', mask_init="ones", mask_l1=0.0, mask_l2=0.0, seed=20)
    exp_object.add_experiment(new_exp)
    exp_object.make_mask(experiment_name+'_tl', overwrite_model=True, overwrite_mask=True, transfer="new_layer")
    exp_object.plot_mask(experiment_name+'_tl', colorbar=False)
    library = ba.BestAnalogs(experiment_name+'_tl', library_name=experiment_name)
    library.get_all_soi_analogs(soi_model, soi_member, tether=tether, savename=None, quant=None, mask_type='weights', leadtime=None)

experiment_settings are:
{'wus_lt1_best_global_early_tl150': {'experiment_name': 'wus_lt1_best_global_early_tl150', 'variable_list': ['tas'], 'analog_models': ['BEST2008'], 'soi_models': ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'], 'leadtime': 1, 'split': {'analog': 1, 'training': 3, 'validation': 2}, 'region': 'western us', 'val_models': ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'], 'target_var': ['tas'], 'window': False, 'trajectory': 1, 'resolutions': [2.7672718509152645, 2.8125], 'time_range': False, 'norm_data': True, 'baseline': [1961, 1990], 'remove_trend': False, 'norm_weights': True, 'transfer_exp': 'wus_lt1_best_global_early', 'mask': False, 'prediction_nodes': [], 'prediction_activation': 'elu', 'learning_rate': 0.0001, 'loss': 'mse', 'patience': 500, 'min_delta': 0.005, 'batch_size': 64, 'val_batch_size': 25

100%|██████████| 173/173 [01:20<00:00,  2.16it/s]


saving to: /Users/mafern/Desktop/mask-analog-projections/saved_analogs/wus_lt1_best_global_early_tl150/BEST_mem0_teth2_library_wus_lt1_best_global_early.pickle
experiment_settings are:
{'wus_lt2_best_global_early_tl150': {'experiment_name': 'wus_lt2_best_global_early_tl150', 'variable_list': ['tas'], 'analog_models': ['BEST2008'], 'soi_models': ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'], 'leadtime': 2, 'split': {'analog': 1, 'training': 3, 'validation': 2}, 'region': 'western us', 'val_models': ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'], 'target_var': ['tas'], 'window': False, 'trajectory': 1, 'resolutions': [2.7672718509152645, 2.8125], 'time_range': False, 'norm_data': True, 'baseline': [1961, 1990], 'remove_trend': False, 'norm_weights': True, 'transfer_exp': 'wus_lt2_best_global_early', 'mask': False, 'prediction

100%|██████████| 173/173 [01:19<00:00,  2.17it/s]


saving to: /Users/mafern/Desktop/mask-analog-projections/saved_analogs/wus_lt2_best_global_early_tl150/BEST_mem0_teth2_library_wus_lt2_best_global_early.pickle
experiment_settings are:
{'wus_lt3_best_global_early_tl150': {'experiment_name': 'wus_lt3_best_global_early_tl150', 'variable_list': ['tas'], 'analog_models': ['BEST2008'], 'soi_models': ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'], 'leadtime': 3, 'split': {'analog': 1, 'training': 3, 'validation': 2}, 'region': 'western us', 'val_models': ['CMCC-CM2-HR4', 'MIROC-ES2H', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'CMCC-ESM2'], 'target_var': ['tas'], 'window': False, 'trajectory': 1, 'resolutions': [2.7672718509152645, 2.8125], 'time_range': False, 'norm_data': True, 'baseline': [1961, 1990], 'remove_trend': False, 'norm_weights': True, 'transfer_exp': 'wus_lt3_best_global_early', 'mask': False, 'prediction

100%|██████████| 173/173 [01:18<00:00,  2.19it/s]


... (11 outputs left out)

100%|██████████| 173/173 [01:24<00:00,  2.04it/s]


saving to: /Users/mafern/Desktop/mask-analog-projections/saved_analogs/wus_lt9_best_global_early_tl150/BEST_mem0_teth2_library_wus_lt9_best_global_early.pickle
experiment_settings are:
{'wus_lt10_best_global_early_tl150': {'experiment_name': 'wus_lt10_best_global_early_tl150', 'variable_list': ['tas'], 'analog_models': ['BEST2008'], 'soi_models': ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'MRI-ESM2-0'], 'leadtime': 10, 'split': {'analog': 1, 'training': 3, 'validation': 2}, 'region': 'western us', 'val_models': ['CMCC-CM2-HR4', 'MIROC-ES2L', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2H', 'CAMS-CSM1', 'GISS-E2-2-G', 'ACCESS-CM2', 'MRI-ESM2-0'], 'target_var': ['tas'], 'window': False, 'trajectory': 1, 'resolutions': [2.7672718509152645, 2.8125], 'time_range': False, 'norm_data': True, 'baseline': [1961, 1990], 'remove_trend': False, 'norm_weights': True, 'transfer_exp': 'wus_lt10_best_global_early', 'mask': False, 'pred

100%|██████████| 173/173 [01:24<00:00,  2.04it/s]


saving to: /Users/mafern/Desktop/mask-analog-projections/saved_analogs/wus_lt10_best_global_early_tl150/BEST_mem0_teth2_library_wus_lt10_best_global_early.pickle


### 2. create the settings for a new experiment

In [ ]:
# print out the settings
# print(exp_object.make_exp_dictionary.__doc__)
# print('---------------------------------------------------------')
# this lists all the regions available (you can add new regions to regions.json)
# exp_object.list_regions

In [ ]:
experiment_name = "wus_lt5_best_global1"
all_models = ['GISS-E2-2-G', 'NorESM2-LM', 'ACCESS-ESM1-5', 'CanESM5-1', 'CMCC-CM2-HR4', 'CESM2', 'AWI-ESM-1-1', 'CAMS-CSM1', 'CMCC-ESM2', 'CESM2-WACCM', 'MPI-ESM1-2-HR', 'AWI-CM-1-1', 'MIROC', 'MPI-ESM1-2-LR', 'GISS-E2-1-G', 'CNRM-CM6-1', 'CMCC-CM2-SR5', 'MRI-ESM2-0', 'MIROC-ES2H', 'CanESM5-CanOE', 'ACCESS-CM2', 'BCC-ESM1', 'UKESM1-1-LL', 'UKESM1-0-LL', 'CanESM5', 'BCC-CSM2', 'TaiESM', 'NorESM2-MM', 'MIROC-ES2L']
obs = ['ERA5', 'BEST']

In [207]:
# now, define a new experiment with all the settings you want to change
new_exp = exp_object.make_exp_dictionary(experiment_name, variable_list=['tas'], analog_models=all_models, soi_models=all_models, leadtime=5, split=[5, 3, 2], region='western us', val_models=all_models, target_var=['tas'], window=False, trajectory=1, resolutions=1, time_range=False, norm_data=True, baseline=[1961, 1990], remove_trend=False, norm_weights=True, transfer_exp=None, mask=False, prediction_nodes=[], prediction_activation='elu', learning_rate=0.001, loss='mse', patience=50, min_delta=0.005, batch_size=64, val_batch_size=2500, max_epochs=5000, mask_activation='relu', mask_init="ones", mask_l1=0.0, mask_l2=0.0, seed=20)

# this adds the experiment to the json file
exp_object.add_experiment(new_exp)

experiment_settings are:
{'wus_lt5_best_global1': {'experiment_name': 'wus_lt5_best_global1', 'variable_list': ['tas'], 'analog_models': ['CMCC-CM2-HR4', 'MIROC-ES2H', 'BCC-CSM2', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2L'], 'soi_models': ['CMCC-CM2-HR4', 'MIROC-ES2H', 'BCC-CSM2', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2L'], 'leadtime': 5, 'split': {'analog': 5, 'training': 3, 'validation': 2}, 'region': 'western us', 'val_models': ['CMCC-CM2-HR4', 'MIROC-ES2H', 'BCC-CSM2', 'NorESM2-MM', 'GISS-E2-1-G', 'MIROC-ES2L'], 'target_var': ['tas'], 'window': False, 'trajectory': 1, 'resolutions': [2.7672718509152645, 2.8125], 'time_range': False, 'norm_data': True, 'baseline': [1961, 1990], 'remove_trend': False, 'norm_weights': True, 'transfer_exp': None, 'mask': False, 'prediction_nodes': [], 'prediction_activation': 'elu', 'learning_rate': 0.001, 'loss': 'mse', 'patience': 50, 'min_delta': 0.005, 'batch_size': 64, 'val_batch_size': 2500, 'max_epochs': 5000, 'mask_activation': 'relu', 'mask_ini

### 3. train (and plot) a mask based on the settings

In [208]:
# NOTE: this may need to be run in a terminal when the amount of data is large
exp_object.make_mask(experiment_name, overwrite_model=True, overwrite_mask=True)

loading variables: tas
from models: CMCC-CM2-HR4 MIROC-ES2H BCC-CSM2 NorESM2-MM GISS-E2-1-G MIROC-ES2L
more members requested than available. requested: 5, available: 1
using 1 members from CMCC-CM2-HR4
found preprocessed data for CMCC-CM2-HR4
more members requested than available. requested: 5, available: 3
using 3 members from MIROC-ES2H
found preprocessed data for MIROC-ES2H
more members requested than available. requested: 5, available: 3
using 3 members from BCC-CSM2
found preprocessed data for BCC-CSM2
more members requested than available. requested: 5, available: 2
using 2 members from NorESM2-MM
found preprocessed data for NorESM2-MM
found preprocessed data for GISS-E2-1-G
found preprocessed data for MIROC-ES2L
#################################################################
processing analog input dataset
#################################################################
baselining on [1961, 1990] for variables: tas
-------------------------------------------------------
stan

<xarray.Dataset>
Dimensions:  (lat: 65, lon: 128)
Coordinates:
  * lat      (lat) float32 -88.62 -85.85 -83.08 -80.31 ... 82.95 85.72 88.49
  * lon      (lon) float32 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
Data variables:
    tas      (lat, lon) float32 5.298e-05 5.341e-05 ... 0.0001093 0.0001096

In [209]:
# plot the mask
exp_object.plot_mask(experiment_name, colorbar=False)
# make a gif of the training
# exp_object.make_mask_gif(experiment_name)

### 4. generate (or select a previously generated) analog library

In [210]:
# model_list is to set the models in the library, n_members sets how many (-1 uses all)
# library_name can be the name of a previous experiment and will use that library
# library = ba.BestAnalogs(experiment_name, library_name=None, model_list=all_models+['ERA5'], n_members=-1)
# library = ba.BestAnalogs(experiment_name, library_name='wus_lt1')

library = ba.BestAnalogs(experiment_name, library_name=None, model_list=all_models+['BEST'], n_members=-1)

loading all available members, using 1 members from model: CMCC-CM2-HR4
loading variables: tas
from models: CMCC-CM2-HR4
found preprocessed data for CMCC-CM2-HR4
#################################################################
processing CMCC-CM2-HR4 input dataset
#################################################################
baselining on [1961, 1990] for variables: tas
-------------------------------------------------------
-------------------------------------------------------
#################################################################
processing CMCC-CM2-HR4 target dataset
#################################################################
extracting region(s): western us
western us found in regions.json
-------------------------------------------------------
baselining on [1961, 1990] for variables: tas
-------------------------------------------------------
-------------------------------------------------------
loading all available members, using 3 members from model: 

### 5. find the best analogs for a given state of interest (model, member)

In [211]:
# this ranks the analogs in the library for this SOI
# this may also need to be run in the terminal
tether = 2
soi_model = 'BEST'
soi_member = 0
library.get_all_soi_analogs(soi_model, soi_member, tether=tether, savename=None, quant=None, mask_type='weights', leadtime=None)

# for global or regional masks
# library.get_all_soi_analogs(soi_model, soi_member, tether=tether, savename='global', quant=None, mask_type='global', leadtime=None)
# library.get_all_soi_analogs(soi_model, soi_member, tether=tether, savename='regional', quant=None, mask_type='regional', leadtime=None)

100%|██████████| 173/173 [01:05<00:00,  2.65it/s]


saving to: /Users/mafern/Desktop/mask-analog-projections/saved_analogs/wus_lt5_best_global1/BEST_mem0_teth2.pickle


### 6. plot the results

In [ ]:
# suite of plots
tether = 2
soi_model = 'BEST'
soi_member = 0
library = pa.PlotAnalogs(experiment_name, soi_model, soi_member, tether, 'tas', library_name=None, fig_path=None)

In [ ]:
n_analogs = 3
roll_years = 7
max_year = 2000

# plot prediction, library range, model mean, perfect analogs
library.plot_prediction_suite(n_analogs, time_range=None, savename=None)
# here is the distribution of how often each model was picked as a top 10 analog
library.plot_model_distribution(n_analogs, time_range=None, savename=None)
# or plot how often each model is picked as a function of time
library.plot_model_times(n_analogs, time_range=None, savename=None)
# plot the selected years versus predicted year
library.plot_selected_times(n_analogs, roll_years, time_range=None, savename=None)
# plot prediction constrained to only select analogs from early than max_year
library.plot_constrained_prediction(n_analogs, max_year, time_range=None, savename=None)
# plot the prediction as a distribution
library.plot_dist_prediction(n_analogs, savename=None, time_range=None)